# G05 — Baseline Keras latent diffusion model trained from scratch

This notebook documents generator **G05 (`05_ldm_basic_fromscratch`)**, the original Keras latent-diffusion baseline. A custom variational autoencoder (VAE) compresses 512 × 512 grayscale mammograms to a 64 × 64 × 4 latent representation, and a class-conditional U-Net is trained from random initialization in that latent space.

G05 has a deliberately limited scientific role. Checkpoint diagnostics are computed for both labels, but the final generation, adaptive filtering, and validation comparison in this notebook target the **positive class only**. G05 is therefore a descriptive positive-class experiment, not a complete two-class candidate and not the selected from-scratch generator used by the downstream classifier study. Its results must not be interpreted as evidence from an independent balanced generator.

The principal locations are:

- shared real images and metadata: `data/processed/`;
- shared traditional augmentation: `data/real_augmented/`;
- models, checkpoints, latent caches, and execution logs: `experiments/diffusers/05_ldm_basic_fromscratch/`;
- metrics, figures, and sustainability records: `results/2_diffusers/05_ldm_basic_fromscratch/`;
- reusable training and evaluation implementations: `notebooks/utility/`.

Expensive stages are designed to be restartable. Each phase is a boolean that starts `False`, while the VAE and LDM helpers independently detect completed model artifacts. Heavy retraining and full regeneration therefore require an explicit decision. These safeguards support reproducible reruns, but they do not turn the exploratory G05 design into confirmatory evidence.


## 1. Runtime bootstrap and accelerator discovery

The following cell locates the repository, adds the notebook utilities to the Python path, discovers an XLA `libdevice` installation, and reports the available NVIDIA hardware. Training and generation run in child processes: the training helper receives a dedicated environment, whereas generation can resolve one or more devices automatically through the shared parallel-generation utilities.

Device visibility is an execution concern rather than part of the scientific configuration. A portable run should inherit or externally define the available devices; no machine-specific GPU UUID belongs in the notebook. The printed inventory and XLA paths are diagnostics only and are not model-selection variables or scientific artifacts.


In [ ]:
# === Unified notebooks/ bootstrap ===
# Works from the project root and every subdirectory under notebooks/.
import sys as _sys
from pathlib import Path as _Path

def _find_mammo_root():
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if _candidate.name == "MammoDiffusion":
            return _candidate
        if (_candidate / "data").is_dir() and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("MammoDiffusion root not found from " + str(_Path.cwd()))

PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

# === End unified bootstrap ===

import os
import subprocess
import sys
from pathlib import Path

CUDA_ROOT_OVERRIDE = os.environ.get("MAMMODIFFUSION_CUDA_ROOT")
CUDA_ROOT_CANDIDATES = [
    Path(CUDA_ROOT_OVERRIDE).expanduser() if CUDA_ROOT_OVERRIDE else None,
    Path(os.environ["CONDA_PREFIX"]) if os.environ.get("CONDA_PREFIX") else None,
    Path(sys.prefix) if sys.prefix else None,
]

CUDA_ROOT = None
for candidate in CUDA_ROOT_CANDIDATES:
    if candidate is None:
        continue
    libdevice_path = candidate / "nvvm" / "libdevice" / "libdevice.10.bc"
    if libdevice_path.exists():
        CUDA_ROOT = candidate.resolve()
        os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={CUDA_ROOT}"
        break

if CUDA_ROOT is None:
    print("libdevice.10.bc not found. Set MAMMODIFFUSION_CUDA_ROOT if XLA requires an explicit CUDA root.")

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.stdout.strip():
        print("Available physical GPUs:")
        print(result.stdout.strip())
except FileNotFoundError:
    print("nvidia-smi is unavailable in this environment.")

print("CUDA_ROOT:", CUDA_ROOT if CUDA_ROOT is not None else "not set")
print("XLA_FLAGS:", os.environ.get("XLA_FLAGS", ""))

# Inherit CUDA visibility by default; optionally override it for training subprocesses.
TRAIN_GPU_VISIBLE_DEVICES = os.environ.get("MAMMODIFFUSION_TRAIN_GPU")

def training_subprocess_env():
    env = os.environ.copy()
    if TRAIN_GPU_VISIBLE_DEVICES is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(TRAIN_GPU_VISIBLE_DEVICES)
    return env

# Multi-GPU generation does not affect training.
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None

def add_generation_parallel_args(command):
    if PARALLEL_GENERATION:
        command.extend(["--generation-gpus", GENERATION_GPU_DEVICES])
        if GENERATION_MAX_WORKERS is not None:
            command.extend(["--max-generation-workers", str(GENERATION_MAX_WORKERS)])
    else:
        command.extend(["--generation-gpus", "off"])
    return command

## 2. Software environment

Dependency installation is disabled by default through `INSTALL_DEPENDENCIES=False`. When intentionally enabled while preparing a new environment, the cell installs the numerical, TensorFlow, PyTorch, and metric packages used by G05. It creates no scientific result. A publication rerun should retain the default in an already provisioned environment and record resolved versions whenever the dependency set is changed.


In [ ]:
# The project environment should normally be provisioned from requirements.txt.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    %pip install -q --upgrade pip
    %pip install -q pandas numpy matplotlib scikit-learn pillow gdown tensorflow scikit-image scipy psutil codecarbon torch torchvision torchmetrics torch-fidelity prdc
else:
    print('Dependencies already provisioned; installation skipped.')

## 3. Project layout and idempotent execution policy

The next two cells resolve the project root without assuming a workstation-specific path, create the canonical experiment/result directories, and verify that the VAE and LDM helper programs are present. They then declare one boolean per phase for training, evaluation, generation, filtering, and the RAW-vs-filtered validation.

Every phase starts disabled, so a Run All neither trains from scratch nor rebuilds an image pool. The historical validation sweep stays off: `RUN_EVALUATION_PHASE = False` delegates to the notebook's explicit selection-file and checkpoint checks, and an intentional recomputation also needs `EVAL_FORCE_RECOMPUTE=True` because the legacy sweep images predate per-directory generation manifests. Filtering remains content-aware. The training helpers add a second safeguard: an existing valid VAE is reused, and an existing terminal LDM model causes training to exit without overwriting it.

Separating `experiments/` (large resumable state) from `results/` (tables, plots, and energy records) preserves source context and makes accidental regeneration easier to detect.


In [ ]:
from pathlib import Path
import shutil
import sys

PROJECT_NAME = "MammoDiffusion"
EXPERIMENT_NAME = "diffusers/05_ldm_basic_fromscratch"

# When needed on Colab/Drive:
# PROJECT_ROOT_OVERRIDE = Path("/content/drive/MyDrive/MammoDiffusion")
PROJECT_ROOT_OVERRIDE = None

def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {root}")
        return root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name:
            return candidate
        has_notebooks = (candidate / "notebooks").exists() or (candidate / "notebooks").exists()
        if ((candidate / "data").exists() and has_notebooks) or ((candidate / ".git").exists() and has_notebooks):
            return candidate

    for candidate in [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
    ]:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not locate the MammoDiffusion root. "
        "Run the notebook from the repository or set PROJECT_ROOT_OVERRIDE."
    )

PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / ("notebooks" if (PROJECT_ROOT / "notebooks").is_dir() else "notebooks")
UTILITY_DIR = NOTEBOOKS_DIR / "utility" if (NOTEBOOKS_DIR / "utility").is_dir() else NOTEBOOKS_DIR
DATA_DIR = PROJECT_ROOT / "data"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
ARCHIVES_DIR = DATA_DIR / "archives"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT_NAME
ACTIVE_G05_FILTERED_DIR = EXPERIMENT_DIR / "synthetic_filtered_positive"

MODELS_DIR = EXPERIMENT_DIR / "models"
CHECKPOINTS_DIR = EXPERIMENT_DIR / "checkpoints_ldm"
LATENTS_DIR = EXPERIMENT_DIR / "latents"
LOGS_DIR = EXPERIMENT_DIR / "logs"
RESULTS_STAGE_NAME = "2_diffusers/05_ldm_basic_fromscratch"
RESULTS_DIR = PROJECT_ROOT / "results" / RESULTS_STAGE_NAME
RESULTS_PLOTS_DIR = RESULTS_DIR / "plots"
RESULTS_METRICS_DIR = RESULTS_DIR / "metrics"
RESULTS_ECOTRACKER_DIR = RESULTS_DIR / "ecotracker"
HELPER_PATH = UTILITY_DIR / "train_ldm.py"
VAE_HELPER_PATH = UTILITY_DIR / "train_vae.py"

for directory in [
    DATA_PROCESSED_DIR,
    ARCHIVES_DIR,
    EXPERIMENT_DIR,
    MODELS_DIR,
    CHECKPOINTS_DIR,
    LATENTS_DIR,
    LOGS_DIR,
    RESULTS_DIR,
    RESULTS_PLOTS_DIR,
    RESULTS_METRICS_DIR,
    RESULTS_ECOTRACKER_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED_DIR:", DATA_PROCESSED_DIR)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("HELPER_PATH:", HELPER_PATH)
print("VAE_HELPER_PATH:", VAE_HELPER_PATH)

if not HELPER_PATH.exists():
    raise FileNotFoundError(f"Helper not found: {HELPER_PATH}")
if not VAE_HELPER_PATH.exists():
    raise FileNotFoundError(f"VAE helper not found: {VAE_HELPER_PATH}")

In [ ]:
# EXPLICIT_PHASE_FLAGS_V1
# One boolean per phase. Every flag is False, so an ordinary Run All reads the
# artifacts already on disk and reports them: it never retrains and never
# regenerates. To do real work, set the flags for the phases you intend to run
# and execute the notebook top to bottom. Flags are independent -- filtering can
# be redone without regenerating the pool it selects from.
#
#   RUN_TRAINING_PHASE    train the model (hours of GPU)
#   RUN_GENERATION_PHASE  sample a full RAW image pool from the selected checkpoint
#   RUN_EVALUATION_PHASE  score checkpoints and record the selection
#   RUN_FILTER_PHASE      re-run the adaptive filter over the existing RAW pool
#   RUN_VALIDATION_PHASE  RAW-vs-filtered comparison; keeps its own content-aware cache
#
# Leaving a flag False asserts that the phase's artifact is already complete.
# The cells below check that claim and raise if it does not hold, rather than
# reporting a number they did not verify.
RUN_TRAINING_PHASE = False
RUN_GENERATION_PHASE = False
RUN_EVALUATION_PHASE = False  # the saved selection is consumed, not recomputed
RUN_FILTER_PHASE = False
RUN_VALIDATION_PHASE = False

## 4. Shared preprocessed dataset

This section validates the expected `train`, `val`, and `test` directory structure for labels 0 and 1 and checks that the split metadata files exist. If the processed dataset is absent, the code downloads the shared archive, validates that it is a ZIP file, extracts it, and reconstructs missing CSV metadata from the directory layout. Existing complete data are reused without extraction.

Training utilities consume only the training and validation partitions; the test partition is reserved for final classifier evaluation. The split counts printed here are a structural audit, not proof of patient-level independence, which is established upstream during preprocessing. Rebuilding metadata from filenames is a recovery path and should be checked against the canonical preprocessing manifest before a publication rerun.


In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

import pandas as pd

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

# Shared ZIP: train/0, train/1, val/0, val/1, test/0, test/1, metadata/*.csv
PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
PROCESSED_ZIP_PATH = ARCHIVES_DIR / "processed.zip"
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_SPLITS = ["train", "val"]
EXPECTED_LABELS = ["0", "1"]
REQUIRED_METADATA = ["all_processed.csv", "train.csv", "val.csv"]

def count_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return 0
    return sum(
        1
        for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )

def get_split_label_counts(processed_dir):
    processed_dir = Path(processed_dir).resolve()
    rows = []
    for split in EXPECTED_SPLITS:
        for label in EXPECTED_LABELS:
            rows.append(
                {
                    "split": split,
                    "label": int(label),
                    "folder": str(processed_dir / split / label),
                    "n_images": count_images(processed_dir / split / label),
                }
            )
    return pd.DataFrame(rows)

def processed_dataset_ready(processed_dir):
    processed_dir = Path(processed_dir).resolve()
    if not processed_dir.exists():
        return False
    counts_df = get_split_label_counts(processed_dir)
    ready = (counts_df["n_images"] > 0).all()
    if not ready and any((processed_dir / split).exists() for split in EXPECTED_SPLITS):
        print("Preprocessed dataset found, but its layout is incomplete:")
        print(counts_df[["split", "label", "n_images"]].to_string(index=False))
    return bool(ready)

def metadata_complete(processed_dir):
    metadata_dir = Path(processed_dir) / "metadata"
    return all((metadata_dir / name).exists() for name in REQUIRED_METADATA)

def parse_filename_metadata(image_path):
    parts = Path(image_path).stem.split("_")
    patient_id = parts[0] if len(parts) >= 1 else Path(image_path).stem
    image_id = parts[1] if len(parts) >= 2 else Path(image_path).stem
    laterality = parts[2] if len(parts) >= 3 else "unknown"
    view = parts[3] if len(parts) >= 4 else "unknown"
    return patient_id, image_id, laterality, view

def rebuild_metadata_from_folders(processed_dir):
    processed_dir = Path(processed_dir).resolve()
    metadata_dir = processed_dir / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for split in EXPECTED_SPLITS:
        for label_name in EXPECTED_LABELS:
            label_dir = processed_dir / split / label_name
            label = int(label_name)
            for image_path in sorted(label_dir.rglob("*")):
                if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                patient_id, image_id, laterality, view = parse_filename_metadata(image_path)
                rows.append(
                    {
                        "patient_id": str(patient_id),
                        "image_id": str(image_id),
                        "laterality": str(laterality),
                        "view": str(view),
                        "label": label,
                        "cancer": label,
                        "patient_label": label,
                        "split": split,
                        "source": "real",
                        "original_path": "",
                        "processed_path": str(image_path),
                    }
                )
    if not rows:
        raise FileNotFoundError(f"No valid image found under {processed_dir}")

    df = pd.DataFrame(rows).sort_values(
        ["split", "label", "patient_id", "image_id"]
    ).reset_index(drop=True)
    df.to_csv(metadata_dir / "all_processed.csv", index=False)
    for split in EXPECTED_SPLITS:
        df[df["split"] == split].reset_index(drop=True).to_csv(
            metadata_dir / f"{split}.csv",
            index=False,
        )
    print("Metadata CSV ricostruiti in:", metadata_dir)
    print(pd.crosstab(df["split"], df["label"]))

def ensure_metadata(processed_dir):
    if metadata_complete(processed_dir):
        print("Metadata CSV files already present:", Path(processed_dir) / "metadata")
        return
    print("Metadata CSV missing: li ricostruisco da train/val/test/0-1.")
    rebuild_metadata_from_folders(processed_dir)

def download_processed_zip():
    if PROCESSED_ZIP_PATH.exists():
        if zipfile.is_zipfile(PROCESSED_ZIP_PATH):
            print("Archive already present; skipping download:", PROCESSED_ZIP_PATH)
            return
        print("The existing archive is invalid; downloading it again.")
        PROCESSED_ZIP_PATH.unlink()
    print("Downloading processed.zip from Google Drive...")
    gdown.download(id=PROCESSED_DRIVE_ID, output=str(PROCESSED_ZIP_PATH), quiet=False)

    if not PROCESSED_ZIP_PATH.exists() or PROCESSED_ZIP_PATH.stat().st_size == 0:
        raise RuntimeError("Download failed. Check the Drive file's sharing settings.")
    if not zipfile.is_zipfile(PROCESSED_ZIP_PATH):
        raise RuntimeError(f"The downloaded file is not a valid ZIP archive: {PROCESSED_ZIP_PATH}")
    print("Download complete:", PROCESSED_ZIP_PATH)

def clear_processed_dir():
    DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    for item in DATA_PROCESSED_DIR.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()

def extract_processed_zip():
    print("Extracting processed.zip under:", DATA_PROCESSED_DIR)
    clear_processed_dir()
    with zipfile.ZipFile(PROCESSED_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_PROCESSED_DIR)

def prepare_processed_dataset():
    if processed_dataset_ready(DATA_PROCESSED_DIR):
        print("Preprocessed dataset already present; skipping download and extraction.")
        ensure_metadata(DATA_PROCESSED_DIR)
        return DATA_PROCESSED_DIR.resolve()

    PROCESSED_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    download_processed_zip()
    extract_processed_zip()

    if not processed_dataset_ready(DATA_PROCESSED_DIR):
        counts_df = get_split_label_counts(DATA_PROCESSED_DIR)
        print(counts_df[["split", "label", "folder", "n_images"]].to_string(index=False))
        raise FileNotFoundError(
            "processed.zip was extracted, but its layout is not the expected one. "
            "The ZIP must directly contain train/0, train/1, val/0, val/1, "
            "test/0, test/1 e metadata/*.csv."
        )

    ensure_metadata(DATA_PROCESSED_DIR)
    return DATA_PROCESSED_DIR.resolve()

DATASET_ROOT = prepare_processed_dataset()
print("DATASET_ROOT:", DATASET_ROOT)
print(get_split_label_counts(DATASET_ROOT)[["split", "label", "n_images"]].to_string(index=False))

## 5. Custom VAE training and reuse

`train_vae.py` trains the G05 grayscale VAE for at most 100 epochs with batch size 8 and learning rate $10^{-4}$. The helper constructs deterministic mild augmentations of positive training images, monitors reconstruction quality on validation data, applies early stopping, and stores the best encoder and decoder as `models/vae_encoder_best.keras` and `models/vae_decoder_best.keras`. Training curves, reconstruction examples, summary metrics, and sustainability records are written under the G05 result directory; the complete console stream is retained in `logs/vae_train.log`.

If both best-model files already exist, the helper exits without retraining unless its explicit force flag is supplied. The notebook also verifies that both outputs are non-empty after the subprocess succeeds. This protects completed state from a routine Run All, although VAE selection remains part of the same G05 development workflow and is not an independent evaluation.


In [ ]:
import os
import subprocess
import time

VAE_EPOCHS = 100
VAE_BATCH_SIZE = 8
VAE_LR = 1e-4

vae_log_path = LOGS_DIR / "vae_train.log"
vae_cmd = [
    sys.executable,
    str(VAE_HELPER_PATH),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--epochs", str(VAE_EPOCHS),
    "--batch-size", str(VAE_BATCH_SIZE),
    "--learning-rate", str(VAE_LR),
]

env = training_subprocess_env()
print("VAE training command:")
print(" ".join(vae_cmd))
print("Log:", vae_log_path)
print("CUDA_VISIBLE_DEVICES:", env.get("CUDA_VISIBLE_DEVICES", ""))
print("XLA_FLAGS:", env.get("XLA_FLAGS", ""))

if RUN_TRAINING_PHASE:
    with open(vae_log_path, "w", encoding="utf-8") as log_file:
        proc_vae = subprocess.Popen(
            vae_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_vae = proc_vae.pid
    print(f"VAE training started - PID {pid_vae}")

    with open(vae_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_vae.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)

        for line in log_file:
            print(line, end="", flush=True)

    print("Process VAE exited with return code:", proc_vae.returncode)
    if proc_vae.returncode != 0:
        raise RuntimeError(f"VAE training failed. Check the log: {vae_log_path}")
else:
    print("VAE training disabled: RUN_TRAINING_PHASE = False.")

for filename in ["vae_encoder_best.keras", "vae_decoder_best.keras"]:
    path = MODELS_DIR / filename
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"VAE was not produced correctly: {path}")
    print("OK:", path)

### 5.1 Read-only VAE diagnostics

The following cell displays `vae_metrics.png` and `vae_reconstruction.png` from `results/2_diffusers/05_ldm_basic_fromscratch/plots/`. It never invokes the trainer and reports a missing file rather than synthesizing replacement evidence. These panels are qualitative and optimization diagnostics; they can reveal unstable training or poor reconstruction, but they do not measure the fidelity or clinical validity of generated mammograms.


In [ ]:
from IPython.display import display
from PIL import Image as PILImage

for plot_name in ["vae_metrics.png", "vae_reconstruction.png"]:
    plot_path = RESULTS_PLOTS_DIR / plot_name
    if plot_path.exists():
        display(PILImage.open(plot_path))
        print("Plot:", plot_path)
    else:
        print("Plot not found:", plot_path)

## 6. Class-conditional LDM training

`train_ldm.py` fits the baseline conditional U-Net in the custom-VAE latent space for a target of 80,000 optimizer steps, writing periodic Keras checkpoints every 7,000 steps and progress messages every 20 steps. The subprocess log is stored at `logs/ldm_train.log`; the helper also writes training plots, sustainability data, and `training_manifest.json`, which records the model parameterization and terminal model path.

`RESUME_FROM_LATEST` is `False` in this historical G05 configuration. Nevertheless, the helper detects an existing terminal model and exits instead of training over it; an intentional continuation requires enabling resume with a larger target. When resume is enabled, model weights, the inferred global step, and the loss history are restored, but the run should not be assumed bitwise identical to uninterrupted training. Checkpoint-loss behavior is diagnostic only: the official checkpoint used for synthesis is selected later on validation generation metrics.


In [ ]:
import os
import subprocess
import time

TOTAL_STEPS = 80_000
CHECKPOINT_EVERY = 7_000
LOG_EVERY = 20
RESUME_FROM_LATEST = False

log_path = LOGS_DIR / "ldm_train.log"
cmd = [
    sys.executable,
    str(HELPER_PATH),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--total-steps", str(TOTAL_STEPS),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--log-every", str(LOG_EVERY),
]

if RESUME_FROM_LATEST:
    cmd.append("--resume-from-latest")

env = training_subprocess_env()
print("Training command:")
print(" ".join(cmd))
print("Log:", log_path)
print("CUDA_VISIBLE_DEVICES:", env.get("CUDA_VISIBLE_DEVICES", ""))
print("XLA_FLAGS:", env.get("XLA_FLAGS", ""))

if RUN_TRAINING_PHASE:
    with open(log_path, "w", encoding="utf-8") as log_file:
        proc = subprocess.Popen(
            cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_ldm = proc.pid
    print(f"LDM training started - PID {pid_ldm}")

    with open(log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)

        for line in log_file:
            print(line, end="", flush=True)

    print("Process training exited with return code:", proc.returncode)
    if proc.returncode != 0:
        raise RuntimeError(f"Training LDM failed. Check the log: {log_path}")
else:
    print("LDM training disabled: RUN_TRAINING_PHASE = False.")

terminal_models = sorted(CHECKPOINTS_DIR.glob("ldm_unet_final_step*.keras"))
if not terminal_models:
    raise FileNotFoundError("No terminal G05 LDM checkpoint is available.")
print("Terminal LDM checkpoint:", terminal_models[-1])

### 6.1 Read-only LDM optimization diagnostics

This cell displays the previously saved `ldm_metrics.png` and performs no model loading or optimization. The curve is useful for identifying divergence, plateaus, or discontinuities after a resumed run. It cannot by itself select a generator checkpoint because lower denoising loss need not correspond to better sample fidelity, coverage, or absence of memorization.


In [ ]:
from IPython.display import display
from PIL import Image as PILImage

plot_path = RESULTS_PLOTS_DIR / "ldm_metrics.png"
if plot_path.exists():
    display(PILImage.open(plot_path))
    print("Plot:", plot_path)
else:
    print("Plot not found:", plot_path)

## 7. Validation-only checkpoint sweep

The `evaluate_ldm.py` sweep discovers eligible checkpoints, generates 100 images per class with 100 reverse-diffusion steps and guidance scale 1.5, and computes FID, Inception Score, and PRDC statistics against validation references.

The saved validation checkpoint selection is reused by default. Set `RUN_EVALUATION_PHASE=True` with `EVAL_FORCE_RECOMPUTE=True` to run the validation sweep again; a recomputation writes new operational manifests, so stored results are never silently reinterpreted. Sweep directories written before per-directory generation manifests existed are not rerun automatically.

The evaluation directory contains checkpoint-specific samples, `checkpoint_metrics.json`, `sweep_results.csv`, `best_checkpoint.json`, and comparison figures. The notebook validates `best_checkpoint.json` and resolves the recorded periodic checkpoint through the current project root. That periodic checkpoint, rather than the mutable convenience copy `ldm_unet_best_eval.keras`, is consumed downstream.

These metrics use a small finite sample and generic Inception features. They are suitable for within-protocol screening, not calibrated estimates of diagnostic realism. Reusing the stored decision verifies checkpoint consumption but is not a metric recomputation.

### 7.1 Checkpoint decision rule and scope

The primary checkpoint rule minimizes positive-class FID, with positive-class Inception Score used only as a tie-breaker. Metrics for label 0 are retained as auxiliary stability diagnostics but do not determine the G05 choice. This asymmetric rule matches the notebook's goal of producing positive images for class augmentation.

Consequently, the selected checkpoint is a **positive-class descriptive choice**, not evidence that G05 is a balanced two-class generator. Selection uses validation data only. Because multiple checkpoints are screened with only 100 generated samples per class, small numerical differences should not be over-interpreted.


In [ ]:
# IDEMPOTENT_GUARD_V1:evaluation
# Keep configuration available to downstream cells even when this phase is skipped.
import json

EVAL_MIN_STEP = 1_000
N_GEN_PER_CLASS = 100
EVAL_SAMPLE_STEPS = 100
EVAL_GUIDANCE_SCALE = 1.5
EVAL_MINI_BATCH = 1
EVAL_INCEPTION_BATCH = 8
EVAL_INCEPTION_WEIGHTS = "imagenet"
EVAL_FORCE_RECOMPUTE = False
EVAL_MODE = "both"
EVAL_DECODE_ON_CPU = False
EVAL_ECO_TRACK = True
EVAL_FORCE_RECOMPUTE = False  # ignore cached metrics and reevaluate

if RUN_EVALUATION_PHASE:
    import os
    import subprocess
    import time

    eval_log_path = LOGS_DIR / "ldm_evaluate.log"
    eval_cmd = [
        sys.executable,
        str(UTILITY_DIR / "evaluate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--mode", EVAL_MODE,
        "--min-step", str(EVAL_MIN_STEP),
        "--n-gen-per-class", str(N_GEN_PER_CLASS),
        "--sample-steps", str(EVAL_SAMPLE_STEPS),
        "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
        "--mini-batch", str(EVAL_MINI_BATCH),
        "--inception-batch", str(EVAL_INCEPTION_BATCH),
        "--inception-weights", EVAL_INCEPTION_WEIGHTS,
    ]

    if EVAL_FORCE_RECOMPUTE:
        eval_cmd.append("--force-recompute")
    if EVAL_DECODE_ON_CPU:
        eval_cmd.append("--decode-on-cpu")
    if EVAL_ECO_TRACK:
        eval_cmd.append("--eco-track")

    add_generation_parallel_args(eval_cmd)
    env = os.environ.copy()
    print("Evaluation command:")
    print(" ".join(eval_cmd))
    print("Log:", eval_log_path)
    print("CUDA_VISIBLE_DEVICES:", env.get("CUDA_VISIBLE_DEVICES", ""))
    print("XLA_FLAGS:", env.get("XLA_FLAGS", ""))

    with open(eval_log_path, "w", encoding="utf-8") as log_file:
        proc_eval = subprocess.Popen(
            eval_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_eval = proc_eval.pid
    print(f"Evaluation avviata - PID {pid_eval}")

    with open(eval_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_eval.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)

        for line in log_file:
            print(line, end="", flush=True)

    print("Process evaluation exited with return code:", proc_eval.returncode)
    if proc_eval.returncode != 0:
        raise RuntimeError(f"Evaluation failed. Check the log: {eval_log_path}")

G05_SELECTION_PATH = EXPERIMENT_DIR / "evaluation" / "best_checkpoint.json"
if not G05_SELECTION_PATH.is_file():
    raise FileNotFoundError(f"Frozen G05 selection is unavailable: {G05_SELECTION_PATH}")
with G05_SELECTION_PATH.open(encoding="utf-8") as handle:
    G05_SELECTION = json.load(handle)
recorded_checkpoint = Path(G05_SELECTION["best_checkpoint"])
G05_SELECTED_CHECKPOINT = CHECKPOINTS_DIR / recorded_checkpoint.name
if not G05_SELECTED_CHECKPOINT.is_file() or G05_SELECTED_CHECKPOINT.stat().st_size == 0:
    raise FileNotFoundError(
        f"Frozen G05 periodic checkpoint is unavailable: {G05_SELECTED_CHECKPOINT}"
    )
print("Frozen G05 selection:", G05_SELECTION["best_checkpoint_id"])
print("Portable checkpoint path:", G05_SELECTED_CHECKPOINT)

### 7.2 PRDC trajectories across checkpoints

This read-only analysis loads `evaluation/sweep_results.csv`, orders checkpoints by training step, and plots precision, recall, density, and coverage separately for labels 0 and 1. A vertical marker identifies the validation-selected checkpoint, and the figure is saved as `checkpoint_prdc_comparison.png`.

The panel complements the FID-based decision by exposing non-monotonic behavior, loss of coverage, or an apparent precision–recall trade-off. It does not change the registered selection rule, and no single PRDC curve is sufficient to diagnose mode collapse without sample and nearest-neighbor inspection.


In [ ]:
import json

import numpy as np
import matplotlib.pyplot as plt

sweep_df = (
    pd.read_csv(EXPERIMENT_DIR / "evaluation" / "sweep_results.csv")
    .sort_values("checkpoint_order")
    .reset_index(drop=True)
)
with open(EXPERIMENT_DIR / "evaluation" / "best_checkpoint.json", encoding="utf-8") as handle:
    best_ckpt = json.load(handle)

best_id = best_ckpt["best_checkpoint_id"]
x = np.arange(len(sweep_df))
labels = sweep_df["checkpoint_id"].astype(str).tolist()
best_positions = sweep_df.index[sweep_df["checkpoint_id"].astype(str) == best_id].tolist()
best_x = best_positions[0] if best_positions else None

metric_specs = [
    ("precision_0", "precision_1", "Precision"),
    ("recall_0", "recall_1", "Recall"),
    ("density_0", "density_1", "Density"),
    ("coverage_0", "coverage_1", "Coverage"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for ax, (col_0, col_1, title) in zip(axes.ravel(), metric_specs):
    ax.plot(x, sweep_df[col_0].astype(float), marker="o", linewidth=1.8, label="class 0")
    ax.plot(x, sweep_df[col_1].astype(float), marker="s", linewidth=1.8, label="class 1")
    if best_x is not None:
        ax.axvline(best_x, color="crimson", linestyle="--", linewidth=1.4)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="best")

fig.suptitle(f"Checkpoint PRDC trend across the sweep - BEST: {best_id}", fontsize=14)
output_path = RESULTS_PLOTS_DIR / "checkpoint_prdc_comparison.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output_path)

## 8. Positive-class generation, filtering, and reverse-diffusion audit

Using the immutable periodic checkpoint validated from `evaluation/best_checkpoint.json`, `generate_ldm.py --mode both` completes a target pool of 2,722 raw positive samples, applies the adaptive mammography filter, retains 1,361 samples, and renders reverse-diffusion diagnostics. The registered G05 positive pool is the portable experiment-relative directory `experiments/diffusers/05_ldm_basic_fromscratch/synthetic_filtered_positive/`; it is passed explicitly so no similarly named G06 or shared-data pool can be substituted. The combined helper is entered when either generation or filtering requires work. Raw generation is resumable at the image-index level: existing readable PNGs are preserved, missing or corrupt indices are regenerated, and worker reservations avoid duplicate work during multi-GPU execution. Filter caches and manifests bind outputs to the model, sampling configuration, and source-image signatures.

Raw samples and filtered images remain under the G05 experiment directory; metrics, filter reports, figures, logs, and EcoTracker events are written to the corresponding results tree. The filter is a deterministic image-quality screen, not a clinical detector. It can improve apparent fidelity while reducing diversity or coverage, so raw and filtered representations must remain distinguishable in all downstream interpretation.


In [ ]:
# IDEMPOTENT_GUARD_V1:generation
# Keep configuration available to validation/final-evaluation cells when generation is skipped.
GEN_MODE = None  # resolved by the phase flags below
GEN_N_RAW = 2722
GEN_N_SELECTED = 1361
GEN_TARGET_LABEL = 1
GEN_BATCH_SIZE = 1
GEN_SAMPLE_STEPS = 100
GEN_GUIDANCE_SCALE = 1.5
GEN_MODEL_PATH = G05_SELECTED_CHECKPOINT
GEN_ECO_TRACK = True

if RUN_GENERATION_PHASE or RUN_FILTER_PHASE:
    GEN_MODE = "both" if RUN_GENERATION_PHASE else "filter"
    import os
    import subprocess
    import time

    gen_log_path = LOGS_DIR / "ldm_generate.log"
    gen_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(GEN_MODEL_PATH),
        "--mode", GEN_MODE,
        "--n-raw", str(GEN_N_RAW),
        "--n-selected", str(GEN_N_SELECTED),
        "--target-label", str(GEN_TARGET_LABEL),
        "--filtered-dir", str(ACTIVE_G05_FILTERED_DIR),
        "--batch-size", str(GEN_BATCH_SIZE),
        "--sample-steps", str(GEN_SAMPLE_STEPS),
        "--guidance-scale", str(GEN_GUIDANCE_SCALE),
    ]

    if GEN_ECO_TRACK:
        gen_cmd.append("--eco-track")

    add_generation_parallel_args(gen_cmd)
    env = os.environ.copy()
    print("Generation command:")
    print(" ".join(gen_cmd))
    print("Log:", gen_log_path)
    print("CUDA_VISIBLE_DEVICES:", env.get("CUDA_VISIBLE_DEVICES", ""))
    print("XLA_FLAGS:", env.get("XLA_FLAGS", ""))

    with open(gen_log_path, "w", encoding="utf-8") as log_file:
        proc_gen = subprocess.Popen(
            gen_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_gen = proc_gen.pid
    print(f"Generation avviata - PID {pid_gen}")

    with open(gen_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_gen.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)

        for line in log_file:
            print(line, end="", flush=True)

    print("Process generation exited with return code:", proc_gen.returncode)
    if proc_gen.returncode != 0:
        raise RuntimeError(f"Generation failed. Check the log: {gen_log_path}")

### 8.1 Adaptive-filter rejection profile

This read-only cell loads `synthetic_filter_summary.json`, aggregates the recorded rejection reasons, and saves `synthetic_filter_reject_reasons.png`. The categories include excessive empty area, excessive foreground, low contrast, fragmented masks, and contact with image borders.

Counts provide an operational audit of why samples were excluded; they are not mutually equivalent measures of pathology or realism. A change in their distribution may indicate a model or preprocessing shift and should trigger manifest and sample review before cached filtered outputs are accepted.


In [ ]:
import json

import matplotlib.pyplot as plt

with open(RESULTS_METRICS_DIR / "synthetic_filter_summary.json", encoding="utf-8") as handle:
    filter_summary = json.load(handle)

reject_counts = filter_summary["reject_counts"]
n_raw = filter_summary["n_raw"]
n_accepted = filter_summary["n_accepted"]
n_selected = filter_summary["n_selected"]

reasons = list(reject_counts.keys())
counts = [reject_counts[reason] for reason in reasons]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(reasons, counts, color="#c0392b")
ax.bar_label(bars, padding=3)
ax.set_title(f"Filter rejection reasons - {n_raw} raw, {n_accepted} accepted, {n_selected} selected")
ax.set_ylabel("Rejected images")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

output_path = RESULTS_PLOTS_DIR / "synthetic_filter_reject_reasons.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output_path)
print("Counts:", reject_counts)

### 8.2 Qualitative filter diagnostics

The next cell displays the already generated accepted/rejected sample panel and the score-distribution panel. These figures compare filter scores and simple foreground statistics with real references and are read-only artifacts of the filtering stage.

The examples are illustrative rather than randomly sampled inferential evidence. They help detect gross threshold failures, but visual plausibility and agreement with handcrafted quality features do not establish lesion authenticity, privacy safety, or clinical utility.


In [ ]:
from IPython.display import display
from PIL import Image as PILImage

for plot_name in ["synthetic_filter_sample.png", "synthetic_filter_distribution.png"]:
    plot_path = RESULTS_PLOTS_DIR / plot_name
    if plot_path.exists():
        display(PILImage.open(plot_path))
        print("Plot:", plot_path)
    else:
        print("Plot not found:", plot_path)

### 8.3 Count-matched raw-versus-filtered validation analysis

`generate_ldm.py --mode validate` compares three positive-class representations against the validation set: the complete raw pool, a deterministic seed-42 raw subset matched to the filtered sample count, and the 1,361 images from the registered experiment-local G05 pool. The filtered directory is supplied explicitly, and the content-aware signature invalidates any cached result produced from another generator's pool. The count-matched subset reduces the direct sample-size confound when FID, Inception Score, and PRDC are compared. Results are written to `raw_vs_filtered_validation.csv` and its companion JSON under the class-scoped metrics directory, with compatibility fallbacks for older layouts.

This is an additional GPU computation and is independently resumable through its output signatures. Its purpose is to quantify the validation-set filter trade-off, not to tune the filter. Improvements in fidelity-like metrics may coexist with worse recall or coverage and should be reported together.


In [ ]:
# IDEMPOTENT_GUARD_V1:generation
# RAW-versus-filtered validation has its own content-aware cache and does not
# depend on generation already being complete.
if RUN_VALIDATION_PHASE:
    import os
    import subprocess
    import time

    VALIDATE_N_RAW = GEN_N_RAW
    VALIDATE_N_SELECTED = GEN_N_SELECTED
    VALIDATE_TARGET_LABEL = GEN_TARGET_LABEL
    VALIDATE_BALANCED_SEED = 42
    VALIDATE_INCEPTION_BATCH = EVAL_INCEPTION_BATCH
    VALIDATE_IS_SPLITS = 10
    VALIDATE_KNN_K = 3
    VALIDATE_ECO_TRACK = True

    validate_log_path = LOGS_DIR / "ldm_validate.log"
    validate_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--mode", "validate",
        "--n-raw", str(VALIDATE_N_RAW),
        "--n-selected", str(VALIDATE_N_SELECTED),
        "--target-label", str(VALIDATE_TARGET_LABEL),
        "--filtered-dir", str(ACTIVE_G05_FILTERED_DIR),
        "--balanced-seed", str(VALIDATE_BALANCED_SEED),
        "--inception-batch", str(VALIDATE_INCEPTION_BATCH),
        "--is-splits", str(VALIDATE_IS_SPLITS),
        "--knn-k", str(VALIDATE_KNN_K),
        "--results-stage-name", RESULTS_STAGE_NAME,
    ]
    if VALIDATE_ECO_TRACK:
        validate_cmd.append("--eco-track")

    add_generation_parallel_args(validate_cmd)
    env = os.environ.copy()
    print("Validation command:")
    print(" ".join(validate_cmd))
    print("Log:", validate_log_path)

    with open(validate_log_path, "w", encoding="utf-8") as log_file:
        proc_validate = subprocess.Popen(
            validate_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_validate = proc_validate.pid
    print(f"Validation started - PID {pid_validate}")

    with open(validate_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_validate.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)
        for line in log_file:
            print(line, end="", flush=True)

    print("Process validate exited with return code:", proc_validate.returncode)
    if proc_validate.returncode != 0:
        raise RuntimeError(f"Validate failed. Check the log: {validate_log_path}")

### 8.4 Visualization of the filtering trade-off

This read-only section converts `raw_vs_filtered_validation.csv` into a direct FID comparison, before/after metric bars, a PRDC radar chart, and direction-aware percentage changes. The plots are saved as `raw_vs_filtered_fid.png`, `filter_before_after.png`, `filter_prdc_radar.png`, and `filter_oriented_percent_delta.png`.

Direction-aware signs make lower-is-better FID comparable with higher-is-better metrics, but percentage changes can be unstable near zero. The figures summarize the validation evidence; they do not authorize replacing raw with filtered data outside the representation policy recorded by the benchmark.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

raw_vs_filtered_candidates = [
    RESULTS_METRICS_DIR / "positive" / "raw_vs_filtered_validation.csv",
    RESULTS_METRICS_DIR / "raw_vs_filtered_validation.csv",  # retained compatibility metric
]
raw_vs_filtered_path = next((path for path in raw_vs_filtered_candidates if path.exists()), None)
assert raw_vs_filtered_path is not None, (
    f"File not found in any expected path: {raw_vs_filtered_candidates}. "
    "Run the validation cell first."
)

df_rvf = pd.read_csv(raw_vs_filtered_path).set_index("dataset")
balanced_name = next(name for name in df_rvf.index if name.startswith("raw_balanced"))

# 1. FID: complete raw set versus filtered set
fid_order = ["raw_complete", "filtered"]
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(fid_order, df_rvf.loc[fid_order, "FID"], color=["#777777", "#2e8b57"])
ax.set_title("Validation FID - complete raw vs filtered")
ax.set_ylabel("FID")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
output_path = RESULTS_PLOTS_DIR / "raw_vs_filtered_fid.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", output_path)

raw_row = df_rvf.loc[balanced_name]
filtered_row = df_rvf.loc["filtered"]

def metric_pair(metric):
    return float(raw_row[metric]), float(filtered_row[metric])

# 2. Before/after: each metric retains its own scale
metric_specs = [
    ("FID", "FID - lower is better"),
    ("IS_mean", "Inception Score - higher indicates greater diversity"),
    ("precision", "Precision PRDC - higher is better"),
    ("recall", "Recall PRDC - higher indicates greater coverage"),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
for axis, (metric, title) in zip(axes.flat, metric_specs):
    raw_value, filtered_value = metric_pair(metric)
    delta = (filtered_value - raw_value) / raw_value * 100
    axis.plot([0, 1], [raw_value, filtered_value], color="#1f77b4", marker="o", linewidth=2.2, markersize=7)
    axis.annotate(
        "",
        xy=(1, filtered_value),
        xytext=(0, raw_value),
        arrowprops={"arrowstyle": "-|>", "color": "#1f77b4", "linewidth": 2.2, "mutation_scale": 13},
    )
    axis.annotate(
        f"{delta:+.1f}%",
        xy=(1, filtered_value),
        xytext=(7, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color="#1f77b4",
    )
    axis.set_xticks([0, 1], ["RAW (balanced)", "Filtered"])
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.25)
fig.suptitle("Effect of adaptive filtering on the LDM - positive class", fontsize=15)
output_path = RESULTS_PLOTS_DIR / "filter_before_after.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", output_path)

# 3. Radar PRDC
prdc_metrics = ["precision", "recall", "density", "coverage"]
radar_angles = np.linspace(0, 2 * np.pi, len(prdc_metrics), endpoint=False).tolist()
radar_angles += radar_angles[:1]
raw_values = [float(raw_row[metric]) for metric in prdc_metrics]
filtered_values = [float(filtered_row[metric]) for metric in prdc_metrics]
radial_limit = max(1.0, max(raw_values + filtered_values) * 1.1)

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True}, constrained_layout=True)
for label, values, color in [("RAW (balanced)", raw_values, "#777777"), ("Filtered", filtered_values, "#2e8b57")]:
    closed_values = values + values[:1]
    ax.plot(radar_angles, closed_values, color=color, linewidth=2, label=label)
    ax.fill(radar_angles, closed_values, color=color, alpha=0.18)
ax.set_xticks(radar_angles[:-1], [metric.capitalize() for metric in prdc_metrics])
ax.set_ylim(0, radial_limit)
ax.set_title("PRDC profile before and after adaptive filtering - positive LDM class", pad=20)
ax.legend(loc="lower right", bbox_to_anchor=(1.25, -0.05))
output_path = RESULTS_PLOTS_DIR / "filter_prdc_radar.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", output_path)

# 4. Percentage change oriented toward improvement
delta_metrics = ["FID", "IS_mean", "precision", "recall", "density", "coverage"]
oriented_deltas = []
for metric in delta_metrics:
    raw_value, filtered_value = metric_pair(metric)
    raw_delta = (filtered_value - raw_value) / raw_value * 100
    oriented_deltas.append(-raw_delta if metric == "FID" else raw_delta)

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
y_positions = np.arange(len(delta_metrics))
colors = ["#2e8b57" if value >= 0 else "#c0392b" for value in oriented_deltas]
bars = ax.barh(y_positions, oriented_deltas, color=colors, edgecolor="white")
ax.bar_label(bars, labels=[f"{value:+.1f}%" for value in oriented_deltas], padding=3)
ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(y_positions, ["FID (sign reversed)", "IS_mean", "Precision", "Recall", "Density", "Coverage"])
ax.set_xlabel("Percentage change oriented toward improvement")
ax.set_title("Adaptive-filter delta - positive LDM class")
ax.grid(axis="x", alpha=0.25)
ax.legend(
    handles=[
        Patch(facecolor="#2e8b57", label="Improvement"),
        Patch(facecolor="#c0392b", label="Reduction / trade-off"),
    ],
    loc="best",
)
output_path = RESULTS_PLOTS_DIR / "filter_oriented_percent_delta.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", output_path)

## 10. Inert manual-termination control

This cell is intentionally non-operative so that Run All cannot terminate a training process. It only prints the disabled status. Manual signalling code is retained as commented operational guidance and requires an explicitly verified process identifier; it creates no scientific artifact and should never be used as a substitute for checkpoint-safe interruption.


In [ ]:
# This cell remains inactive so Run All cannot stop training.
# To stop a process manually, uncomment the lines below and set the verified PID.
# import os
# import signal
# os.kill(pid_ldm, signal.SIGTERM)
print("Manual stop disabled for Run All.")

## 12. Experiment artifact inventory

The final cell recursively lists files below `experiments/diffusers/05_ldm_basic_fromscratch/`, reports human-readable sizes, and displays the first 80 entries. This helps verify that checkpoints, latent caches, logs, evaluation products, and manifests were handed off together.

The table is an inventory rather than an integrity check: presence and size do not replace the content signatures in the runtime and generation manifests. G05 should remain labeled as a positive-only descriptive experiment in any archive derived from this listing.


In [ ]:
from pathlib import Path

def human_size(n_bytes):
    n = float(n_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024

rows = []
for path in sorted(EXPERIMENT_DIR.rglob("*")):
    if path.is_file():
        rows.append(
            {
                "path": path.relative_to(EXPERIMENT_DIR).as_posix(),
                "size": human_size(path.stat().st_size),
            }
        )

summary_df = pd.DataFrame(rows)
print("Artifact files:", len(summary_df))
display(summary_df.head(80))